# Lab 02 — Handwritten Text Recognition (HTR/OCR)

> **GraphoLab** | Forensic Graphology Laboratory

**Model:** `microsoft/trocr-large-handwritten` (Hugging Face)  
**Task:** Transcribe handwritten text from images  
**Forensic use case:** Anonymous letters, historical documents, court exhibits

---

## How TrOCR Works

TrOCR is a Transformer-based OCR model that treats handwriting recognition as an image-to-text task:

1. **Encoder (BEiT):** A Vision Transformer that encodes the input image into a sequence of visual feature vectors.
2. **Decoder (RoBERTa):** A language model Transformer that generates the text token-by-token, conditioned on the visual features.

The model was fine-tuned on the **IAM Handwriting Database** — a standard benchmark containing handwritten English sentences from ~650 writers.

> **Note on line segmentation:** TrOCR is optimised for *single-line* images. For multi-line documents, we first segment the image into individual line strips using a horizontal projection profile, then transcribe each line separately. This avoids the hallucinations that occur when the model is shown multiple lines at once.

## Setup


In [ ]:
# Install dependencies if not already installed
# !pip install transformers torch Pillow requests matplotlib

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import requests
from io import BytesIO

import torch
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Load the Model

The model is downloaded from Hugging Face on first run (~1.3 GB) and cached locally in `~/.cache/huggingface/hub`. Subsequent runs load from cache instantly.

In [ ]:
MODEL_NAME = "microsoft/trocr-large-handwritten"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME} ...")
processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print("Model ready.")

## Helper Functions


In [ ]:
import cv2
from spellchecker import SpellChecker

def load_image(source: str | Path) -> Image.Image:
    """Load an image from a file path or URL."""
    if isinstance(source, Path) or (isinstance(source, str) and not source.startswith("http")):
        return Image.open(source).convert("RGB")
    import requests
    from io import BytesIO
    response = requests.get(source, timeout=10)
    return Image.open(BytesIO(response.content)).convert("RGB")


_TROCR_IT_MAP = {
    "january": "gennaio", "february": "febbraio", "march": "marzo",
    "april": "aprile",    "may": "maggio",         "june": "giugno",
    "july": "luglio",     "august": "agosto",      "september": "settembre",
    "october": "ottobre", "november": "novembre",  "december": "dicembre",
    "monday": "lunedì",   "tuesday": "martedì",    "wednesday": "mercoledì",
    "thursday": "giovedì","friday": "venerdì",     "saturday": "sabato",
    "sunday": "domenica",
    "the": "il", "and": "e", "of": "di", "for": "per",
    "with": "con", "from": "da", "by": "da",
}
_spell_it = SpellChecker(language="it")


def correct_italian(text: str) -> str:
    """Post-process TrOCR output for Italian documents."""
    result_lines = []
    for line in text.splitlines():
        tokens = line.split()
        corrected = []
        for token in tokens:
            prefix, suffix, word = "", "", token
            while word and not word[0].isalnum():
                prefix += word[0]; word = word[1:]
            while word and not word[-1].isalnum():
                suffix = word[-1] + suffix; word = word[:-1]
            if not word:
                corrected.append(token); continue

            lower = word.lower()
            if lower in _TROCR_IT_MAP:
                repl = _TROCR_IT_MAP[lower]
                if word[0].isupper():
                    repl = repl.capitalize()
                corrected.append(prefix + repl + suffix); continue

            if word[0].isupper() or not word.isalpha() or len(word) <= 2:
                corrected.append(token); continue
            if lower not in _spell_it:
                suggestion = _spell_it.correction(lower)
                if suggestion and suggestion != lower and suggestion in _spell_it.edit_distance_1(lower):
                    corrected.append(prefix + suggestion + suffix); continue
            corrected.append(token)
        result_lines.append(" ".join(corrected))
    return "\n".join(result_lines)


def segment_lines(pil_img: Image.Image, min_gap: int = 5, pad: int = 6) -> list[Image.Image]:
    """Split a multi-line handwritten image into individual line crops."""
    gray = np.array(pil_img.convert("L"))
    _, binary = cv2.threshold(
        cv2.GaussianBlur(gray, (3, 3), 0), 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU,
    )
    proj = binary.sum(axis=1)
    in_line, start, segments = False, 0, []
    for r, v in enumerate(proj):
        if v > 0 and not in_line:
            in_line, start = True, r
        elif v == 0 and in_line:
            if r - start >= min_gap:
                segments.append((start, r))
            in_line = False
    if in_line:
        segments.append((start, len(proj)))
    if not segments:
        return [pil_img]
    h, w = gray.shape
    return [pil_img.crop((0, max(0, y0 - pad), w, min(h, y1 + pad))) for y0, y1 in segments]


def transcribe(image: Image.Image) -> str:
    """Transcribe a handwritten image (single- or multi-line)."""
    lines = segment_lines(image)
    texts = []
    for line_img in lines:
        pixel_values = processor(images=line_img, return_tensors="pt").pixel_values.to(DEVICE)
        with torch.no_grad():
            generated_ids = model.generate(pixel_values)
        texts.append(processor.batch_decode(generated_ids, skip_special_tokens=True)[0])
    return correct_italian("\n".join(texts))


def show_result(image: Image.Image, text: str, title: str = "TrOCR Transcription") -> None:
    """Display the image with its transcription."""
    fig, (ax_img, ax_text) = plt.subplots(1, 2, figsize=(14, 4),
                                          gridspec_kw={'width_ratios': [1, 1]})
    ax_img.imshow(image, cmap='gray' if image.mode == 'L' else None)
    ax_img.set_title("Input Image", fontsize=12)
    ax_img.axis('off')
    ax_text.text(0.05, 0.5, text, fontsize=13, va='center', wrap=True,
                 fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax_text.set_title("Transcription", fontsize=12)
    ax_text.axis('off')
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Demo 1 — Single Line (Public Sample)

We use a sample image from the IAM dataset hosted publicly for demonstration.


In [ ]:
# Public sample from Hugging Face (IAM handwriting)
SAMPLE_URL = "https://fki.tic.heia-fr.ch/static/img/a01-122-02.jpg"

# Fallback: use a local sample if available
local_sample = Path("../data/samples/handwritten_text_01.png")

if local_sample.exists():
    image = load_image(local_sample)
    print(f"Loaded local sample: {local_sample}")
else:
    # Create a simple synthetic handwriting-like image for demo
    from PIL import ImageDraw, ImageFont
    image = Image.new('RGB', (600, 80), color='white')
    draw = ImageDraw.Draw(image)
    draw.text((10, 20), "The quick brown fox jumps over the lazy dog",
              fill='black')
    print("Using synthetic fallback image.")

transcription = transcribe(image)
print(f"\nTranscription: {transcription}")
show_result(image, transcription)

## Demo 2 — Load Your Own Image

Replace the path below with your own handwritten image:


In [ ]:
# ─── Change this path to your own image ───────────────────────────────────────
USER_IMAGE_PATH = "../data/samples/handwritten_text_01.png"
# ──────────────────────────────────────────────────────────────────────────────

path = Path(USER_IMAGE_PATH)
if path.exists():
    user_image = load_image(path)
    user_transcription = transcribe(user_image)
    print(f"Transcription: {user_transcription}")
    show_result(user_image, user_transcription, title="Your Image — TrOCR Transcription")
else:
    print(f"File not found: {path}")
    print("Please place a handwritten image at the path above and re-run this cell.")

## Demo 3 — Batch Transcription

Useful in forensic pipelines where multiple document images need to be processed:


In [ ]:
samples_dir = Path("../data/samples")
image_files = sorted(samples_dir.glob("handwritten_text_*.png"))

if not image_files:
    print("No handwritten_text_*.png files found in data/samples/")
    print("Add images to data/samples/ and re-run.")
else:
    print(f"Found {len(image_files)} image(s)\n")
    results = []
    for img_path in image_files:
        img = load_image(img_path)
        text = transcribe(img)
        results.append((img_path.name, text))
        print(f"  {img_path.name}: {text}")

    print("\nBatch transcription complete.")

## Evaluation — Character Error Rate (CER)

When a ground-truth transcript is available, we can measure transcription quality with **CER** — the fraction of characters that were incorrectly predicted.


In [ ]:
def cer(reference: str, hypothesis: str) -> float:
    """Compute Character Error Rate (CER) using edit distance."""
    import numpy as np
    r, h = list(reference), list(hypothesis)
    d = np.zeros((len(r) + 1, len(h) + 1), dtype=int)
    for i in range(len(r) + 1):
        d[i][0] = i
    for j in range(len(h) + 1):
        d[0][j] = j
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            cost = 0 if r[i - 1] == h[j - 1] else 1
            d[i][j] = min(d[i-1][j] + 1, d[i][j-1] + 1, d[i-1][j-1] + cost)
    return d[len(r)][len(h)] / max(len(r), 1)


# Example evaluation
ground_truth = "The quick brown fox jumps over the lazy dog"
predicted = transcription  # from Demo 1

score = cer(ground_truth, predicted)
print(f"Ground truth : {ground_truth}")
print(f"Predicted    : {predicted}")
print(f"CER          : {score:.3f}  ({score*100:.1f}% character errors)")

## Forensic Notes

- TrOCR is optimised for **line-level** recognition. For full documents, pre-segment the image into lines before running inference.
- Performance degrades on **unusual scripts**, non-Latin alphabets, or highly stylised writing. Consider domain-specific fine-tuning.
- The model does not preserve **layout** (paragraph breaks, indentation). Use a document layout analyser (e.g., LayoutLM) for structured documents.
- Always cross-check automatic transcriptions against the original image in a legal context.

---

**Next lab →** [03 — Signature Authenticity Verification](03_signature_verification_siamese.ipynb)
